In [64]:
import re
import json
import pandas as pd
import os
from pathlib import Path

In [65]:
import os
os.chdir(r"D:\Study\Programs\trading")

In [66]:
date = "30APR2026"

In [67]:


def extract_snapshots(log_file_path):
    pattern = re.compile(r'WINDOW_SNAPSHOT:\s*(\{.*\})')

    snapshots = []

    with open(log_file_path, 'r') as f:
        for line in f:
            match = pattern.search(line)
            if match:
                try:
                    data = json.loads(match.group(1))
                    snapshots.append(data)
                except json.JSONDecodeError:
                    continue

    return snapshots


def snapshots_to_dataframe(snapshots):
    df = pd.DataFrame(snapshots)

    # Optional: convert time column
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])

    return df


In [68]:
# ---- Usage ----
log_path = Path(f"assets/logs/{date}/streamer.log")


snapshots = extract_snapshots(log_path)
df = snapshots_to_dataframe(snapshots)

In [69]:
df.shape

(476, 17)

In [70]:
df.head()

,time,nifty,atm,ATM-3_CE,ATM-3_PE,ATM-2_CE,ATM-2_PE,ATM-1_CE,ATM-1_PE,ATM_CE,ATM_PE,ATM+1_CE,ATM+1_PE,ATM+2_CE,ATM+2_PE,ATM+3_CE,ATM+3_PE
0,2026-04-30 09:10:05.328,23996.95,24000,NIFTY2650523850CE,NIFTY2650523850PE,NIFTY2650523900CE,NIFTY2650523900PE,NIFTY2650523950CE,NIFTY2650523950PE,NIFTY2650524000CE,NIFTY2650524000PE,NIFTY2650524050CE,NIFTY2650524050PE,NIFTY2650524100CE,NIFTY2650524100PE,NIFTY2650524150CE,NIFTY2650524150PE
1,2026-04-30 09:15:03.331,23971.60,23950,NIFTY2650523800CE,NIFTY2650523800PE,NIFTY2650523850CE,NIFTY2650523850PE,NIFTY2650523900CE,NIFTY2650523900PE,NIFTY2650523950CE,NIFTY2650523950PE,NIFTY2650524000CE,NIFTY2650524000PE,NIFTY2650524050CE,NIFTY2650524050PE,NIFTY2650524100CE,NIFTY2650524100PE
2,2026-04-30 09:15:38.330,23975.30,24000,NIFTY2650523850CE,NIFTY2650523850PE,NIFTY2650523900CE,NIFTY2650523900PE,NIFTY2650523950CE,NIFTY2650523950PE,NIFTY2650524000CE,NIFTY2650524000PE,NIFTY2650524050CE,NIFTY2650524050PE,NIFTY2650524100CE,NIFTY2650524100PE,NIFTY2650524150CE,NIFTY2650524150PE
3,2026-04-30 09:15:38.831,23974.40,23950,NIFTY2650523800CE,NIFTY2650523800PE,NIFTY2650523850CE,NIFTY2650523850PE,NIFTY2650523900CE,NIFTY2650523900PE,NIFTY2650523950CE,NIFTY2650523950PE,NIFTY2650524000CE,NIFTY2650524000PE,NIFTY2650524050CE,NIFTY2650524050PE,NIFTY2650524100CE,NIFTY2650524100PE
4,2026-04-30 09:15:39.330,23978.05,24000,NIFTY2650523850CE,NIFTY2650523850PE,NIFTY2650523900CE,NIFTY2650523900PE,NIFTY2650523950CE,NIFTY2650523950PE,NIFTY2650524000CE,NIFTY2650524000PE,NIFTY2650524050CE,NIFTY2650524050PE,NIFTY2650524100CE,NIFTY2650524100PE,NIFTY2650524150CE,NIFTY2650524150PE


In [71]:
# df.to_excel("option_chain.xlsx")

In [72]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill
import random

# --- Config ---
COLOR_POOL = [
    "FFC7CE", "C6EFCE", "FFEB9C", "BDD7EE", "D9D2E9",
    "FCE4D6", "E2EFDA", "FFF2CC", "DDEBF7", "EAD1DC",

    "F4CCCC", "D9EAD3", "FFF2CC", "CFE2F3", "D9D2E9",
    "FCE5CD", "EAD1DC", "D0E0E3", "F9CB9C", "C9DAF8",

    "EA9999", "B6D7A8", "FFE599", "9FC5E8", "B4A7D6",
    "F6B26B", "D5A6BD", "A2C4C9", "FFD966", "A4C2F4",

    "E06666", "93C47D", "FFD966", "6FA8DC", "8E7CC3",
    "F6B26B", "C27BA0", "76A5AF", "F1C232", "6D9EEB",

    "CC0000", "6AA84F", "F1C232", "3D85C6", "674EA7",
    "E69138", "A64D79", "45818E", "BF9000", "3C78D8",

    "990000", "38761D", "BF9000", "134F5C", "351C75",
    "783F04"
]

# --- Persistent mapping ---
symbol_color_map = {}

def get_color(symbol):
    if symbol not in symbol_color_map:
        # deterministic or random
        color = COLOR_POOL[len(symbol_color_map) % len(COLOR_POOL)]
        symbol_color_map[symbol] = color
    return symbol_color_map[symbol]


def write_colored_excel(df, file_name="output.xlsx"):
    wb = Workbook()
    ws = wb.active

    # Write header
    ws.append(list(df.columns))

    for row_idx, row in df.iterrows():
        excel_row = []

        for col in df.columns:
            value = row[col]
            excel_row.append(value)

        ws.append(excel_row)

        # Apply colors AFTER writing row
        for col_idx, col in enumerate(df.columns, start=1):
            value = row[col]

            if isinstance(value, str) and value.startswith("NIFTY"):
                color = get_color(value)

                fill = PatternFill(
                    start_color=color,
                    end_color=color,
                    fill_type="solid"
                )

                ws.cell(row=row_idx + 2, column=col_idx).fill = fill

    wb.save(file_name)

In [73]:
path_ = Path(f"assets/logs/{date}/option_chain_{date}.xlsx")
path_.parent.mkdir(parents=True, exist_ok=True)
write_colored_excel(df, file_name=path_)

In [74]:
df.columns

Index(['time', 'nifty', 'atm', 'ATM-3_CE', 'ATM-3_PE', 'ATM-2_CE', 'ATM-2_PE',
       'ATM-1_CE', 'ATM-1_PE', 'ATM_CE', 'ATM_PE', 'ATM+1_CE', 'ATM+1_PE',
       'ATM+2_CE', 'ATM+2_PE', 'ATM+3_CE', 'ATM+3_PE'],
      dtype='str')